# 3. checkpoint_resume_scraping.py with Restructuring the code using OOP
## Feature Improvements
- **OOP:** Organized the scraper into a reusable class with separate methods.
- **Dynamic Loading Detection:** Added page-height monitoring with retry tracking to detect when no new content is loaded and stop scrolling after consecutive unsuccessful attempts.

In [31]:
import os
import pandas as pd
import numpy as np
import time
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

In [32]:
class PropertyScraper:
    def __init__(self, url, timeout=5):
        self.url = url

        # Initialize checkpoint-related attributes
        self.property_data = []
        self.processed_ids = set()
        
        self.driver = self._initialize_driver()
        self.wait = WebDriverWait(self.driver, timeout=timeout)

        

    def _initialize_driver(self):
        driver = webdriver.Chrome()
        driver.maximize_window()
        return driver

        

    def _wait_for_page_to_load(self):
        title = self.driver.title
        #within given time 5sec if all page get load i.e complete then good to proceed
        try:
            self.wait.until(
                lambda d: d.execute_script("return document.readyState") == "complete"
            )
        except:
            print(f"The webpage \"{title}\" did not get fully loaded.\n")
        #if webpage load completely in given duration
        else:
            print(f"The webpage \"{title}\" did get fully loaded.\n")

            

    def access_website(self):
        self.driver.get(self.url)
        self._wait_for_page_to_load()

        
    
    def search_properties(self, location):
        """
        - Locate the search bar.
        - Cancel the already selected Mumbai tag, if present.
        - Enter location in the search bar.
        - Select the valid location option from the suggestion list.
        - Click the search button to display the properties.
        """
        #identifying search bar 
        try:
            search_bar = self.wait.until(
                EC.presence_of_element_located((By.XPATH, '//*[@id="keyword"]'))
            )
        except:
            print("Timeout while locating Search Bar.\n")
        else:
            #search and cancel mumbai tag
            try:
                search_bar.click()
                clk = self.wait.until(
                    EC.element_to_be_clickable((By.XPATH, '//*[@id="keyword_autoSuggestSelectedDiv"]/div/div[2]'))
                )
                clk.click()
            except:
                print("mumbai tag not found")
            finally: #enter location in search bar
                search_bar.click()
                search_bar.send_keys(location)
                time.sleep(5)
        
        
        
        #selecting valid option from list
        try:
            valid_option = self.wait.until(
                EC.element_to_be_clickable((By.XPATH, '//*[@id="serachSuggest"]/div[3]'))
            )
        except:
            print("Timeout while locating valid search option.\n")
        else:
            valid_option.click()
            time.sleep(2)
        
        
        
        #click on search button
        try:
            search_button = self.wait.until(
                EC.element_to_be_clickable((By.XPATH, '//*[@id="searchFormHolderSection"]/section/div/div[1]/div[3]/div[4]'))
            )
        except:
            print("Timeout while clicking on \"search\" button.\n")
        else:
            search_button.click()
            self._wait_for_page_to_load()

        
        
    def apply_filters(self, offset):

        """
        Apply property filters on the MagicBricks search results.
        - Open the Property Type filter and uncheck House/Villa allow 
          only all Flat properties appear in results.
        - Open the BHK filter and uncheck 2 BHK and 3 BHK to allow
          properties of all BHK configurations to appear in the results.
        - Open More Filters.
        - Select Ready to Move, Verified Properties, Photos, Videos,
          and RERA Properties.
        - Click "View Selected Properties" to display the filtered results.
        """
        
        # click on property type button
        try:
            property_type_button = self.wait.until(
                EC.presence_of_element_located((By.XPATH, '//*[@id="body"]/div[1]/div/div[2]/div[3]/div/div[1]'))
            )
        except:
            print("Timeout while locating \"property_type_button\" button.\n")
        else:
            self.driver.execute_script(
                "window.scrollBy(0, arguments[0].getBoundingClientRect().top - arguments[1]);",
                property_type_button,
                offset
            )
            time.sleep(2)
        
            # find the element again because DOM may have changed
            property_type_button = self.wait.until(
                EC.element_to_be_clickable((By.XPATH, '//*[@id="body"]/div[1]/div/div[2]/div[3]/div/div[1]'))
            )
        
            property_type_button.click()
            time.sleep(2)
        
        
        
        # unchecked House/Villa
        try:
            house_villa = self.wait.until(
                EC.presence_of_element_located((By.ID, "propResidential_propertyType_10001_10017"))
            )
        except:
            print("Timeout while locating \"House/Villa\" checkbox.\n")
        else:
            if house_villa.is_selected():
                house_villa_label = self.wait.until(
                    EC.element_to_be_clickable((By.XPATH, '//*[@id="body"]/div[1]/div/div[2]/div[3]/div/div[2]/div[1]/div[2]/div[2]/label'))
                )
                house_villa_label.click()
                time.sleep(2)
        
                
        
        #clicking on done
        try:
            click_done = self.wait.until(
                EC.element_to_be_clickable((By.XPATH, '//*[@id="body"]/div[1]/div/div[2]/div[3]/div/div[2]/div[4]'))
            )
        except:
            print("Timeout while locating \"done\".\n")
        else:
            click_done.click()
            time.sleep(2)
        
        
        
        # click on BHK button
        try:
            BHK_button = self.wait.until(
                EC.presence_of_element_located((By.XPATH, '//*[@id="body"]/div[1]/div/div[2]/div[4]/div/div[1]/div'))
            )
        except:
            print("Timeout while locating \"BHK\" button.\n")
        else:
            self.driver.execute_script(
                "window.scrollBy(0, arguments[0].getBoundingClientRect().top - arguments[1]);",
                BHK_button,
                offset
            )
            time.sleep(2)
        
            # find the element again because DOM may have changed
            BHK_button = self.wait.until(
                EC.element_to_be_clickable((By.XPATH, '//*[@id="body"]/div[1]/div/div[2]/div[4]/div/div[1]/div'))
            )
        
            BHK_button.click()
            time.sleep(2)
        
        
        
        # unchecked 2BHK
        try:
            BHK2 = self.wait.until(
                EC.presence_of_element_located((By.ID, "bhk_11701"))
            )
        except:
            print("Timeout while locating \"2 BHK\" checkbox.\n")
        else:
            if BHK2.is_selected():
                BHK2_label = self.wait.until(
                    EC.element_to_be_clickable((By.XPATH, '//*[@id="body"]/div[1]/div/div[2]/div[4]/div/div[2]/div[1]/div[2]/div[2]/label'))
                )
                BHK2_label.click()
                time.sleep(2)
        
                
        
        # unchecked 3BHK
        try:
            BHK3 = self.wait.until(
                EC.presence_of_element_located((By.ID, "bhk_11702"))
            )
        except:
            print("Timeout while locating \"3 BHK\" checkbox.\n")
        else:
            if BHK3.is_selected():
                BHK3_label = self.wait.until(
                    EC.element_to_be_clickable((By.XPATH, '//*[@id="body"]/div[1]/div/div[2]/div[4]/div/div[2]/div[1]/div[2]/div[3]/label'))
                )
                BHK3_label.click()
                time.sleep(2)
        
                
        
        #clicking on done
        try:
            click_done = self.wait.until(
                EC.element_to_be_clickable((By.XPATH, '//*[@id="body"]/div[1]/div/div[2]/div[4]/div/div[2]/div[2]'))
            )
        except:
            print("Timeout while locating \"done\".\n")
        else:
            click_done.click()
            time.sleep(2)
        
        
        # click on More Filters button
        try:
            more_filters_button = self.wait.until(
                EC.presence_of_element_located((By.XPATH, '//*[@id="body"]/div[1]/div/div[2]/div[6]/div/div[1]'))
            )
        except:
            print("Timeout while locating \"more_filters\" button.\n")
        else:
            self.driver.execute_script(
                "window.scrollBy(0, arguments[0].getBoundingClientRect().top - arguments[1]);",
                more_filters_button,
                offset
            )
            time.sleep(2)
        
            # find the element again because DOM may have changed
            more_filters_button = self.wait.until(
                EC.element_to_be_clickable((By.XPATH, '//*[@id="body"]/div[1]/div/div[2]/div[6]/div/div[1]'))
            )
        
            more_filters_button.click()
            time.sleep(2)
        
        
        
        # checked ready to move button
        try:
            ready_to_move_button = self.wait.until(
                EC.presence_of_element_located((By.ID, "possessionStatus_10081"))
            )
        except:
            print("Timeout while locating \"ready_to_move\" button.\n")
        else:
            ready_to_move_label = self.wait.until(
                EC.element_to_be_clickable((By.XPATH, '//*[@id="moreFilter_1"]/div[2]/div[1]/label'))
            )
            ready_to_move_label.click()
            time.sleep(2)
        
                
        # checked verified properties button
        try:
            verified_properties_button = self.wait.until(
                EC.presence_of_element_located((By.ID, "verifiedProperties_Y"))
            )
        except:
            print("Timeout while locating \"verified_properties\" button.\n")
        else:
            verified_properties_label = self.wait.until(
                EC.element_to_be_clickable((By.XPATH, '//*[@id="moreFilter_11"]/div[2]/div/label'))
            )
            verified_properties_label.click()
            time.sleep(2)
        
        
        # checked photos button
        try:
            photos_button = self.wait.until(
                EC.presence_of_element_located((By.ID, "imageVideo_1"))
            )
        except:
            print("Timeout while locating \"photos\" button.\n")
        else:
            photos_label = self.wait.until(
                EC.element_to_be_clickable((By.XPATH, '//*[@id="moreFilter_12"]/div[2]/div[1]/label'))
            )
            photos_label.click()
            time.sleep(2)
        
        
        
        # checked videos button
        try:
            videos_button = self.wait.until(
                EC.presence_of_element_located((By.ID, "imageVideo_2"))
            )
        except:
            print("Timeout while locating \"videos\" button.\n")
        else:
            videos_label = self.wait.until(
                EC.element_to_be_clickable((By.XPATH, '//*[@id="moreFilter_12"]/div[2]/div[2]/label'))
            )
            videos_label.click()
            time.sleep(2)
        
        
        # checked rera properties button
        try:
            rera_properties_button = self.wait.until(
                EC.presence_of_element_located((By.ID, "reraProperties_Y"))
            )
        except:
            print("Timeout while locating \"rera_properties\" button.\n")
        else:
            rera_properties_label = self.wait.until(
                EC.element_to_be_clickable((By.XPATH, '//*[@id="moreFilter_20"]/div[2]/div/label'))
            )
            rera_properties_label.click()
            time.sleep(2)
        
        
        #clicking on view selected properties
        try:
            view_selected_properties_button = self.wait.until(
                EC.element_to_be_clickable((By.XPATH, '//*[@id="moreRefined"]/div[3]/div/div[2]'))
            )
        except:
            print("Timeout while locating \"view_selected_properties\" button.\n")
        else:
            view_selected_properties_button.click()
            time.sleep(3)


    def checpoint_setup(self, checkpoint_file):   
        if os.path.exists(checkpoint_file):
        
            print("\n======================================")
            print("CHECKPOINT FOUND")
            print("======================================")
        
            checkpoint_df = pd.read_csv(checkpoint_file)
        
            # Existing scraped data
            self.property_data = checkpoint_df.to_dict("records")
        
            # Existing property IDs
            self.processed_ids = set(checkpoint_df["property_id"].dropna().astype(str))
        
            print(f"Already scraped: {len(self.processed_ids)} properties")
        
        else:
            print("\n======================================")
            print("NO CHECKPOINT FOUND")
            print("STARTING FROM SCRATCH")
            print("======================================")
        
            self.property_data = []
            self.processed_ids = set()


    def _extract_data_using_find_element(self, card, by, value):
        try:
            return card.find_element(by, value).text.strip()
        except:
            return np.nan

    
    def scrape_webpage(self, checkpoint_file):
        height = self.driver.execute_script("return document.body.scrollHeight")
        no_growth_attempts = 0
        max_no_growth_attempts = 3
        
        while True:
            # scroll to bottom
            self.driver.execute_script(
                "window.scrollTo(0, document.body.scrollHeight)"
            )
        
            time.sleep(3)
        
            # find all property cards
            property_cards = self.driver.find_elements(By.CLASS_NAME, "mb-srp__list")
        
            print("Total cards:", len(property_cards))
        
        
        
            # find new cards using property ID
            new_cards = []
            for card in property_cards:
                try:
                    card_property_id = card.get_attribute("id").strip()
                except:
                    card_property_id = None
        
                # Skip card if property ID is missing
                if not card_property_id:
                    print("Skipping card: Property ID missing")
                    continue
        
                # Skip property if already scraped
                if card_property_id in self.processed_ids:
                    continue
        
                # This is a new property
                new_cards.append(card)
        
            print("New cards:",len(new_cards))
        
        
        
            # process new cards
            for card in new_cards:
        
                # clicking on down arrrow in each property
                try:
                    down_arrow_button = card.find_element(
                        By.CSS_SELECTOR,"div.mb-srp__card__summary__action"
                    )
        
                    self.driver.execute_script(
                        "arguments[0].scrollIntoView({block: 'center'});",down_arrow_button
                    )
        
                    time.sleep(0.5)
        
                    self.driver.execute_script(
                        "arguments[0].click();",down_arrow_button
                    )
        
                    time.sleep(1)
        
                except Exception as e:
                    print("Down arrow not found")
        
        
                # property id
                try:
                    property_id = card.get_attribute("id").strip()
                    print(property_id)
                except:
                    property_id = np.nan
        

                other_features={
                    "property_title" : self._extract_data_using_find_element(card, By.CLASS_NAME, "mb-srp__card--title"),
                    "property_developer" : self._extract_data_using_find_element(card, By.CLASS_NAME, "mb-srp__card__developer--name--highlight"),
                    "property_carpet_area" : self._extract_data_using_find_element(card, By.XPATH,".//div[text()='Carpet Area']/following-sibling::div"),
                    "property_status" : self._extract_data_using_find_element(card, By.XPATH,".//div[text()='Status']/following-sibling::div"),
                    "property_transaction" : self._extract_data_using_find_element(card, By.XPATH,".//div[text()='Transaction']/following-sibling::div"),
                    "property_furnishing" : self._extract_data_using_find_element(card, By.XPATH,".//div[text()='Furnishing']/following-sibling::div"),
                    "property_society" : self._extract_data_using_find_element(card, By.XPATH,".//div[text()='Society']/following-sibling::div"),
                    "property_car_parking" : self._extract_data_using_find_element(card, By.XPATH,".//div[text()='Car Parking']/following-sibling::div"),
                    "property_floor" : self._extract_data_using_find_element(card, By.XPATH,".//div[text()='Floor']/following-sibling::div"),
                    "property_facing" : self._extract_data_using_find_element(card, By.XPATH,".//div[text()='facing']/following-sibling::div"),
                    "property_overlooking" : self._extract_data_using_find_element(card, By.XPATH,".//div[text()='overlooking']/following-sibling::div"),
                    "property_super_area" : self._extract_data_using_find_element(card, By.XPATH,".//div[text()='Super Area']/following-sibling::div"),
                    "property_price" : self._extract_data_using_find_element(card, By.CLASS_NAME,"mb-srp__card__price--amount"),
                    "property_cost_per_sqft" : self._extract_data_using_find_element(card, By.CLASS_NAME,"mb-srp__card__price--size")
                }

                all_features = {
                    "property_id" : property_id,
                    **other_features
                }
        
                self.property_data.append(all_features)
        
                # Data scraped successfully, so add the property ID to processed_ids
                if pd.notna(property_id):
                    self.processed_ids.add(str(property_id))
        
        
            # checkpoint save
            if len(new_cards) > 0:
                df_checkpoint = pd.DataFrame(self.property_data)
                df_checkpoint.to_csv(checkpoint_file, index=False, encoding="utf-8-sig")
                print(f"CHECKPOINT SAVED: {len(df_checkpoint)} properties")
            else:
                print("No new properties found.")
        
        
        
            # Check page height
            new_height = self.driver.execute_script("return document.body.scrollHeight")
            
            print("Old height:", height)
            print("New height:", new_height)
            
            # Check whether the page has grown
            if new_height > height:
                no_growth_attempts = 0
            else:
                no_growth_attempts += 1
            
                print(f"No page growth: {no_growth_attempts}/{max_no_growth_attempts} attempts")
            
            # Stop after 3 consecutive attempts without page growth
            if no_growth_attempts >= max_no_growth_attempts:
                print("\nNo more properties loaded after multiple attempts.")
                break
            
            # Update height for the next iteration
            height = new_height
            

        df = pd.DataFrame(self.property_data)
        df.to_csv("property_data.csv", index=False, encoding="utf-8-sig")
        
        print("\n======================================")
        print("SCRAPING COMPLETED")
        print("======================================")
        
        print(f"Total properties: {len(df)}")
        
        print("Final file: property_data.csv")
        
        print("Checkpoint file: property_data_checkpoint.csv")
        


    def run(self, location, offset, checkpoint_file):
        try:
            self.access_website()
            self.search_properties(location)
            self.apply_filters(offset)
            self.checpoint_setup(checkpoint_file)
            self.scrape_webpage(checkpoint_file)
        finally:
            time.sleep(2)
            self.driver.quit()
    

In [ ]:
if __name__ == "__main__":
    location = "mumbai"
    scraper = PropertyScraper(url = "https://www.magicbricks.com/")
    checkpoint_file = "property_data_checkpoint.csv"
    
    scraper.run(
        location=location,
        offset=100, 
        checkpoint_file = checkpoint_file
    )

The webpage "Real Estate | Property in India | Buy/Sale/Rent Properties | MagicBricks" did get fully loaded.

The webpage "2 BHK Flats in Mumbai - 39815+ 2 BHK Flats for Sale in Mumbai" did get fully loaded.


CHECKPOINT FOUND
Already scraped: 703 properties
Total cards: 60
New cards: 0
No new properties found.
Old height: 26101
New height: 28780
Total cards: 90
New cards: 0
No new properties found.
Old height: 28780
New height: 37272
Total cards: 120
New cards: 0
No new properties found.
Old height: 37272
New height: 49031
Total cards: 150
New cards: 1
cardid86860891
CHECKPOINT SAVED: 814 properties
Old height: 49031
New height: 61088
Total cards: 180
New cards: 1
cardid86858275
CHECKPOINT SAVED: 815 properties
Old height: 61088
New height: 72920
Total cards: 210
New cards: 0
No new properties found.
Old height: 72920
New height: 84874
Total cards: 240
New cards: 1
cardid86858623
CHECKPOINT SAVED: 816 properties
Old height: 84874
New height: 96777
Total cards: 270
New cards: 2
cardid8